# Trade bot experiment, base in AI signals

### Import libraries

In [ ]:
import numpy as np
import pandas as pd
import os
import  datetime
import json
import time



In [ ]:
import os, sys
processing_source_path = os.path.abspath('./../AI/Processing/')
if(processing_source_path not in sys.path):
    sys.path.append(processing_source_path)
from DataLoaderPipeline import scrapingHistoricalData, FeaturesDataGenerator

import  ProcessingPipeline as pp

In [ ]:
import ccxt  # noqa: E402

In [ ]:
exchange = ccxt.binance({
    'apiKey': 'fDr14DLdIMprsM45Z4yVxKVJVNV1OuvGKuK4yx6byiorXKxAEo39gFwBhxXIRlM3',
    'secret': 'Qp1XABhwv6ZXZb39v11JsBy3AaggMC1fM1pjVfkkPBdHkdqS935rMYF4nMumogsU',
    'options': {
        'defaultType': 'future',
    },
})

In [5]:
def fibonacci_levels(low_price, high_price):
    """
    Calculate the Fibonacci retracement and extension levels from the lowest and highest prices.
    
    Parameters:
    low_price (float): The lowest price of the movement.
    high_price (float): The highest price of the movement.
    
    Returns:
    dict: A dictionary containing the Fibonacci retracement and extension levels.
    """
    # Difference between the high price and the low price
    diff = high_price - low_price
    
    # Calculating the retracement levels
    retracement_levels = {
        '23.6%': high_price - (0.236 * diff),
        '38.2%': high_price - (0.382 * diff),
        '50%': high_price - (0.5 * diff),
        '61.8%': high_price - (0.618 * diff),
    }
    
    # Calculating the extension levels
    extension_levels = {
        '100%': high_price + diff,
        '161.8%': high_price + (1.618 * diff),
        '261.8%': high_price + (2.618 * diff),
        '131.8%': high_price + (1.318 * diff)  # Add the '131.8%' extension level
    }

    return {
        'retracement': retracement_levels,
        'extension': extension_levels
    }

def calculate_targets_and_stops_with_fibonacci(entry_price, low_price, high_price):
    """
    Define the profit targets and stop losses based on the Fibonacci levels.
    
    Parameters:
    entry_price (float): The entry price of the order.
    low_price (float): The lowest price of the movement.
    high_price (float): The highest price of the movement.
    
    Returns:
    dict: A dictionary containing the profit target and stop loss.
    """
    # Calculate the Fibonacci levels
    fibonacci = fibonacci_levels(low_price, high_price)
    
    # Optimistic Scenario
    stop_loss_optimistic = fibonacci['retracement']['38.2%']
    target_optimistic = fibonacci['extension']['161.8%']
    
    # Pessimistic Scenario
    stop_loss_pessimistic = fibonacci['retracement']['61.8%']
    target_pessimistic = fibonacci['extension']['100%']
    
    # Neutral Scenario
    stop_loss_neutral = fibonacci['retracement']['50%']
    target_neutral = fibonacci['extension']['131.8%']
    
    return {
        'optimistic': {
            'stop_loss': round(stop_loss_optimistic, 4),
            'target': round(target_optimistic, 4)
        },
        'pessimistic': {
            'stop_loss': round(stop_loss_pessimistic, 4),
            'target': round(target_pessimistic, 4)
        },
        'neutral': {
            'stop_loss': round(stop_loss_neutral, 4),
            'target': round(target_neutral, 4)
        }
    }

In [18]:
def get_avg_price(symbol):
    try:
        quote = exchange.fetch_ticker(symbol)
        ticker_price=quote['info']['lastPrice']
        price=ticker_price

        
    except  Exception as e:
        print('error')
        return{
            "code":"error",
            "message": str(e),
            'ERRO': 'erro'
            }
    
    #return price,ticker_size,quote
    return price

### Set SHORT order 
def futures_order(trade_symbol, Quantity, order_type='long'):

    if order_type =='long':
        side ='buy'
    elif order_type =='short':
        side ='sell'

    order = exchange.create_order(
        symbol=trade_symbol,
        type='market',
        side = side,  # 'buy' para long, 'sell' para short
        amount=Quantity,  # Quantidade de ADA
        params={
            'positionSide': 'BOTH',  # Ou "LONG"/"SHORT" no modo Hedge
            'marginType': 'CROSSED',  # Ou 'ISOLATED'
            'isAutoAddMargin': 'false',  # Mantém a margem fixa
        }
    )
    return order

def analyze_position(position, symbol=None):
    for pos in position:

        if symbol!= None:
            if f'{symbol}USDT'==pos['symbol']:
                side = 'Short' if float(pos['positionAmt']) < 0 else 'Long'
                quantity = abs(float(pos['positionAmt']))
                entry_price = float(pos['entryPrice'])
                break_even_price = float(pos['breakEvenPrice'])
                current_price = float(pos['markPrice'])
                unrealized_profit = float(pos['unRealizedProfit'])
                leverage = float(pos['leverage'])
                margin_type = pos['marginType']
                notional = float(pos['notional'])

                # Calculating the percentage of profit
                if notional != 0:
                    profit_percentage = (unrealized_profit / abs(notional)) * 100
                else:
                    profit_percentage = 0.0

                print(f" Position: {side}")
                print(f" Quantity: {quantity} {symbol}")
                print(f" Entry Price: {entry_price}")
                print(f" Break-Even Price: {break_even_price}")
                print(f" Current Price (Mark): {current_price}")
                print(f" Unrealized Profit: {unrealized_profit:.6f} USDT")
                print(f" Profit Percentage: {profit_percentage*10:.2f}% {'🔺' if profit_percentage >= 0 else '🔻'}")
                print(f" Position Value: {notional:.4f} USDT ")
                print(f" Leverage: {leverage}x")
                print(f" Margin Type: {margin_type}")

def calculate_pnl_structured(trade_open, trade_close):
    """
    Calculate the profit and loss (PnL) of a trade pair and return structured data.

    Parameters:
        trade_open (dict): Trade entry.
        trade_close (dict): Trade exit.

    Returns:
        dict: Structured trade info with PnL, ROI, and metadata.
    """
    def extract(trade):
        return {
            'orderId': trade['info']['orderId'],
            'clientOrderId': trade['info']['clientOrderId'],
            'side': trade['info']['side'],
            'price': float(trade['info']['avgPrice']),
            'qty': float(trade['info']['executedQty']),
            'cost': float(trade['info']['cumQuote']),
            'datetime': datetime.fromtimestamp(trade['timestamp'] / 1000),
            'positionSide': trade['info'].get('positionSide', 'BOTH'),
            'reduceOnly': trade['info'].get('reduceOnly', False),
            'symbol': trade['symbol']
        }

    t1 = extract(trade_open)
    t2 = extract(trade_close)

    # Identify buy and sell trades
    if t1['side'] == 'BUY' and t2['side'] == 'SELL':
        buy, sell = t1, t2
    elif t1['side'] == 'SELL' and t2['side'] == 'BUY':
        buy, sell = t2, t1
    else:
        raise ValueError("One trade must be a BUY and the other a SELL.")

    pnl = sell['price'] * sell['qty'] - buy['price'] * buy['qty']
    roi_pct = (pnl / buy['cost']) * 100
    duration = str(sell['datetime'] - buy['datetime'])

    trade_summary = {
        'symbol': buy['symbol'],
        'entry': {
            'type': buy['side'],
            'price': buy['price'],
            'qty': buy['qty'],
            'datetime': buy['datetime'].isoformat(),
            'orderId': buy['orderId']
        },
        'exit': {
            'type': sell['side'],
            'price': sell['price'],
            'qty': sell['qty'],
            'datetime': sell['datetime'].isoformat(),
            'orderId': sell['orderId']
        },
        'pnl': round(pnl, 6),
        'roi_pct': round(roi_pct, 4),
        'duration': duration
    }

    # Print nicely
    print("💼 Trade Summary:")
    print(f"  Entry: {buy['side']} {buy['qty']} @ {buy['price']} on {buy['datetime']}")
    print(f"  Exit : {sell['side']} {sell['qty']} @ {sell['price']} on {sell['datetime']}")
    print(f"  ⏱️ Duration: {duration}")
    print(f"  💰 PnL: {pnl:.6f} USDT")
    print(f"  📈 ROI: {roi_pct:.4f}%")

    return trade_summary

### Init Parameters 

In [7]:
symbol="ETH"
list_of_models = ['CNN_MultiHead_2D']
used_model= list_of_models[0]
last_timestamp = pd.Timestamp('2020-02-05 16:00:00')

### Set Futures configuration 

In [8]:
margin_type = 'CROSSED'  # ou 'ISOLATED'

leverage = 10  # Exemplo: 20x de alavancagem
exchange.fapiprivate_post_leverage({
    'symbol': f'{symbol}USDT',
    'leverage': leverage
})

{'symbol': 'ETHUSDT', 'leverage': '10', 'maxNotionalValue': '150000000'}

In [ ]:
aa

### get recomendations

In [9]:
def get_recommendation_history(model_name: str, crypto: str):
    file_path = f'./../AI/Classification/Real_Time_Inference/Recommendations/{model_name}_{crypto}_recommendation.csv'
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        history = []
        for index, row in df.iterrows():
            history.append({
                'Date': row['Date'],
                'Time': row['Time'],
                'recommendation': row['recommendation'],
                'percentage': row['percentage'],
                'Price': row['Price'],
            })
        return history
    else:
        return None

In [10]:
def bot_logic(used_model, symbol, list_of_models, last_timestamp):
    if 'CNN' in used_model:
        model_type = 'CNN'

    AI_recomendations = get_recommendation_history(model_name=model_type, crypto=symbol)
    AI_recomendations_df = pd.DataFrame(AI_recomendations)
    AI_recomendations_df.tail(10)

    # get the last model recommendation 
    last_recomendation = AI_recomendations_df.iloc[-1]
    #get the recommendation timestamp
    last_recomendation_timestamp = pd.to_datetime(f"{last_recomendation['Date']} {last_recomendation['Time']}")

    if last_timestamp != last_recomendation_timestamp:
        if last_recomendation['recommendation'] == "Buy":
            # get the model lookback
            checkpoint_filepath = f'./../AI/Classification/Experiments/Cryptos/models/model_{list_of_models[0]}_crypto_{symbol}_best'
            # JSON file
            with open(f'{checkpoint_filepath}/config.json', 'r') as file:
                parameters = json.load(file)

            lookback = parameters['lookback']

            #get the recommendation timestamp
            last_recomendation_timestamp = pd.to_datetime(f"{last_recomendation['Date']} {last_recomendation['Time']}")

            # scraping the last data 
            SHD = scrapingHistoricalData()
            date_now = datetime.datetime.now()
            date_innit = date_now - datetime.timedelta(days=100)
            cryptos_df = SHD.get_crypto_historical_data([symbol], '4h', date_innit, date_now.strftime('%Y-%m-%d'))

            # get idx that is equal to the AI recomendation timestamp
            last_idx = cryptos_df[cryptos_df['Date'] == last_recomendation_timestamp].index.values[0]
            # get the last N values of loockback
            last_recomendation_window = cryptos_df.iloc[last_idx - lookback: last_idx + 1]

            high = last_recomendation_window['Close'].max()
            low = last_recomendation_window['Close'].min()
            middle = last_recomendation_window['Close'].median()
            input_price = last_recomendation_window['Close'].iloc[-1]

            resultados_fibonacci = calculate_targets_and_stops_with_fibonacci(input_price, low, high)
            print(f"**AI Trust percentage:** {AI_recomendations_df['percentage'].iloc[-1]}% ")

            print("-----------------------------------------------------------------------------")
            print("**Cenário Otimista:** ")
            print("-------------------------------")
            print(f" **Stop Loss:** {resultados_fibonacci['optimistic']['stop_loss']}")
            print(f" **Alvo:** {resultados_fibonacci['optimistic']['target']}")
            print("-------------------------------")

            print("\n**Cenário Pessimista:** ")
            print("-------------------------------")
            print(f" **Stop Loss:** {resultados_fibonacci['pessimistic']['stop_loss']}")
            print(f" **Alvo:** {resultados_fibonacci['pessimistic']['target']}")
            print("-------------------------------")

            print("\n**Cenário Neutro:** ")
            print("-------------------------------")
            print(f" **Stop Loss:** {resultados_fibonacci['neutral']['stop_loss']}")
            print(f" **Alvo:** {resultados_fibonacci['neutral']['target']}")
            print("-------------------------------")
            
            # Retornar a última indicação de recomendação, a porcentagem de certeza e os alvos e stops
            return {
                'recomendation': last_recomendation['recommendation'],
                'percentage': AI_recomendations_df['percentage'].iloc[-1],
                'timestamp':last_recomendation_timestamp,
                'targets_and_stops': {
                    'optimistic': {
                        'stop_loss': resultados_fibonacci['optimistic']['stop_loss'],
                        'target': resultados_fibonacci['optimistic']['target']
                    },
                    'pessimistic': {
                        'stop_loss': resultados_fibonacci['pessimistic']['stop_loss'],
                        'target': resultados_fibonacci['pessimistic']['target']
                    },
                    'neutral': {
                        'stop_loss': resultados_fibonacci['neutral']['stop_loss'],
                        'target': resultados_fibonacci['neutral']['target']
                    }
                }
            }
        elif last_recomendation['recommendation'] == "Sell":
            print(f"**AI Trust percentage:** {AI_recomendations_df['percentage'].iloc[-1]}% ")

            print("Checking for Open signal")

            print("Closing open Signal Based AI recommendation ")

            # Retornar a última indicação de recomendação e a porcentagem de certeza
            return {
                'recomendation': last_recomendation['recommendation'],
                'percentage': AI_recomendations_df['percentage'].iloc[-1],
                'timestamp':last_recomendation_timestamp
            }

    # Update the last timestamp
    last_timestamp = last_recomendation_timestamp
    return {
        'recomendation': last_recomendation['recommendation'],
        'percentage': AI_recomendations_df['percentage'].iloc[-1],
        'timestamp':last_recomendation_timestamp
    }

In [11]:
# get the recomendations and the targets and stops 
result = bot_logic(used_model, symbol, list_of_models, last_timestamp)

print(result)

{'recomendation': 'Hold', 'percentage': 0.7322402, 'timestamp': Timestamp('2025-05-10 20:00:00')}


In [12]:
# ler histórico de ações do ativo

#verificar ordens abertas 
# verificar lucro ou prejuizo 
if result['recomendation'] == 'Buy':

    ## Verifica se tem ordem aberta 
    # se sim  avaliar a porcentagem e/ou lucro

    # se não:
    if result['percentage'] > 0.65 and result['percentage'] < 0.8 :
        stop_loss = result['targets_and_stops']['neutral']['stop_loss']
        target = result['targets_and_stops']['neutral']['target']

    elif result['percentage'] >= 0.8 :

        stop_loss = result['targets_and_stops']['optimistic']['stop_loss']
        target = result['targets_and_stops']['optimistic']['target']

    elif result['percentage'] > 0.5 and result['percentage'] <= 0.65 :
        stop_loss = result['targets_and_stops']['pessimistic']['stop_loss']
        target = result['targets_and_stops']['pessimistic']['target']

    print("Comprando...")
    # Simular a compra (não implementado)

    # Verificar Lucro
    if current_price >= target:
        print("Vendendo para realizar lucro...")
        # Simular a venda (não implementado)

    # Verificar Stop Loss
    elif current_price <= stop_loss:
        print("Vendendo para limitar perdas...")
        # Simular a venda (não implementado)

elif result['recomendation'] == 'Sell':

    # verificar se tem ordem aberta
    # verificar lucro ou preju
    # avaliar a porcentagem
    # decidir se vende ou espera mais
    
    # Vender
    print("Vendendo...")
    # Simular a venda (não implementado)

# salvar histórico de ações para o ativo

In [13]:
USD=5.2
price= get_avg_price(f'{symbol}/USDT')
Quantity= round(USD/float(price))
Quantity

0

In [14]:
# get wallet balance
print('Fetching your balance:')
response = exchange.fetch_balance()
wallet=[[moeda, valor] for moeda, valor in response['total'].items() if valor > 0]
print(wallet)

Fetching your balance:
[['USDT', 99.64124806]]


In [15]:
symbol="ADA"

In [16]:
# set order
trade_open=futures_order(f'{symbol}/USDT', Quantity, order_type='short')
print(trade_open)

InvalidOrder: binance amount of ADA/USDT:USDT must be greater than minimum amount precision of 1

In [ ]:
positions[0]

{'symbol': 'ENAUSDT',
 'positionAmt': '27',
 'entryPrice': '0.3663',
 'breakEvenPrice': '0.36648315',
 'markPrice': '0.36660000',
 'unRealizedProfit': '0.00810000',
 'liquidationPrice': '0',
 'leverage': '20',
 'maxNotionalValue': '1600000',
 'marginType': 'cross',
 'isolatedMargin': '0.00000000',
 'isAutoAddMargin': 'false',
 'positionSide': 'BOTH',
 'notional': '9.89820000',
 'isolatedWallet': '0',
 'updateTime': '1746839932455',
 'isolated': False,
 'adlQuantile': '0'}

In [ ]:
symbol

'ADA'

In [19]:
# Get current positions
print('Getting your positions:')
response = exchange.fapiprivatev2_get_positionrisk()

positions = [position for position in response if float(position['positionAmt'])]

if positions != []: 
    # Convert list of positions to DataFrame
    df_positions = pd.DataFrame(positions)
    analyze_position(positions, symbol)

    df_positions

for pos in positions:
    if f'{symbol}USDT'==pos['symbol']:
        price = float(pos['entryPrice'])
        Position_Value = float(pos['notional'])

side_close = 'buy' if float(positions[0]['positionAmt']) < 0 else 'sell'

print("side_profit",side_close)

Getting your positions:
 Position: Long
 Quantity: 7.0 ADA
 Entry Price: 0.7749
 Break-Even Price: 0.77528745
 Current Price (Mark): 0.81564295
 Unrealized Profit: 0.285201 USDT
 Profit Percentage: 49.95% 🔺
 Position Value: 5.7095 USDT 
 Leverage: 10.0x
 Margin Type: cross
side_profit sell


In [ ]:
# scraping the last data 
SHD = scrapingHistoricalData()
date_now = datetime.datetime.now()
date_innit = date_now - datetime.timedelta(days=100)
cryptos_df = SHD.get_crypto_historical_data([symbol], '4h', date_innit, date_now.strftime('%Y-%m-%d'))


high=cryptos_df.iloc[-100:]['Close'].max()
low=cryptos_df.iloc[-100:]['Close'].min()
response_targets=calculate_targets_and_stops_with_fibonacci(float(positions[0]['entryPrice']), low, high)

stop_loss=response_targets['pessimistic']['stop_loss']
target=response_targets['pessimistic']['target']

print("prince",price)
print("target",target)
print("stop_loss",stop_loss)

prince 0.7749
target 0.9404
stop_loss 0.7038


In [ ]:
amount = Quantity
# STOP LOSS
stop_loss_order = exchange.create_order(
    symbol=f'{symbol}/USDT',
    type='STOP_MARKET',
    side=side_close,  # 'sell' para fechar long; 'buy' para fechar short
    amount=amount,
    params={
        'stopPrice': stop_loss,
        'workingType': 'MARK_PRICE',
        'priceProtect': True,
        'reduceOnly': True  # <- ESSENCIAL
    }
)

# TAKE PROFIT
take_profit_order = exchange.create_order(
    symbol=f'{symbol}/USDT',
    type='TAKE_PROFIT_MARKET',
    side=side_close,  # mesmo lado do stop
    amount=amount,
    params={
        'stopPrice': target,
        'workingType': 'MARK_PRICE',
        'priceProtect': True,
        'reduceOnly': True  # <- ESSENCIAL
    }
)



In [ ]:
stop_loss_id = stop_loss_order['info']['orderId']
take_profit_id = take_profit_order['info']['orderId']

In [ ]:
# Verifica o status de ambas
stop_status = exchange.fetch_order(stop_loss_id, f'{symbol}/USDT')['status']
tp_status = exchange.fetch_order(take_profit_id, f'{symbol}/USDT')['status']

if stop_status == 'closed':
    print("🚨 Stop Loss executado. Cancelando o Take Profit.")
    try:
        exchange.cancel_order(take_profit_id, f'{symbol}/USDT')
    except Exception as e:
        print("Erro ao cancelar TP:", e)

if tp_status == 'closed':
    print("✅ Take Profit executado. Cancelando o Stop Loss.")
    try:
        exchange.cancel_order(stop_loss_id, f'{symbol}/USDT')
    except Exception as e:
        print("Erro ao cancelar SL:", e)


In [ ]:
last_timestamp
symbol
Position_Value
side

price
target
stop_loss

side_close
stop_loss_id
take_profit_id

'open'

In [ ]:
import csv

# Supondo que você tenha uma lista de dicionários com as informações
dados = [
    {"last_timestamp": "valor1", "symbol": "valor2", "Position_Value": "valor3", "side": "valor4",
     "price": "valor5", "target": "valor6", "stop_loss": "valor7", "side_close": "valor8",
     "stop_loss_id": "valor9", "take_profit_id": "valor10"},
    # Outros dados...
]

with open('dados_trade.csv', 'w', newline='') as csvfile:
    fieldnames = ["last_timestamp", "symbol", "Position_Value", "side", "price", "target", "stop_loss", "side_close", "stop_loss_id", "take_profit_id"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    
    writer.writeheader()
    for dado in dados:
        writer.writerow(dado)

In [ ]:
def monitor_and_cancel_opposite(exchange, symbol, stop_order_id, tp_order_id):
    while True:
        orders = exchange.fetch_open_orders(symbol)
        open_order_ids = [order['id'] for order in orders]

        if stop_order_id not in open_order_ids:
            # Stop loss foi executado, cancela o take profit
            try:
                exchange.cancel_order(tp_order_id, symbol)
                print("Stop executado. Take Profit cancelado.")
            except Exception as e:
                print("Erro ao cancelar TP:", e)
            break

        elif tp_order_id not in open_order_ids:
            # Take profit foi executado, cancela o stop loss
            try:
                exchange.cancel_order(stop_order_id, symbol)
                print("Take Profit executado. Stop cancelado.")
            except Exception as e:
                print("Erro ao cancelar SL:", e)
            break

        time.sleep(2)  # espera antes de verificar de novo


Erro: 'binance' object has no attribute 'private_post_position_side_dual'


In [ ]:
# Run the simulation in an infinite loop
while True:
    try:
    
       last_timestamp = bot_logic(used_model, symbol, list_of_models, last_timestamp)

    except Exception as e:
        # Print any exceptions
        print(e)
    # Wait for 20 minutes (1200 seconds) before checking again
    time.sleep(1200)